In [1]:
!pip install transformers datasets peft accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
import json
import argparse
from typing import List, Dict
from inspect import signature

import torch
from datasets import Dataset
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

In [3]:

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [4]:
def load_and_inspect(path: str) -> None:
    ds = load_dataset("json", data_files=path, split="train")
    print(f"Loaded dataset from {path!s}: {len(ds)} examples")
    print("Columns:", ds.column_names)

    if len(ds) > 0:
        print("\nFirst record:")
        first = ds[0]
        for k, v in first.items():
            print(f"- {k}: {v}")

    # Show only question and answer
    keep = [c for c in ds.column_names if c in ("question", "answer")]
    subset = ds.remove_columns([c for c in ds.column_names if c not in keep])
    print("\nFirst QA pair:")
    print(subset[0])
    return subset

def tokenize_function(examples, tokenizer, max_length):
    # <|im_start|>system ... <|im_end|>
    # <|im_start|>user ... <|im_end|>
    # <|im_start|>assistant ... <|im_end|>
    samples = []
    for example in examples:
        system_prompt = "<|im_start|>system ... <|im_end|>\n"
        user_prompt = f"<|im_start|>user {example['question']} <|im_end|>\n<|im_start|>assistant\n"
        label_prompt = f"{example['answer']} <|im_end|>"

        instruction_part = tokenizer(
            system_prompt + user_prompt, add_special_tokens=False
        )
        response_part = tokenizer(label_prompt, add_special_tokens=False)

        input_ids = instruction_part["input_ids"] + response_part["input_ids"]
        attention_mask = (
            instruction_part["attention_mask"] + response_part["attention_mask"]
        )
        labels = [-100] * len(instruction_part["input_ids"]) + response_part[
            "input_ids"
        ]

        # 长度截断（左截断，尽量保留结尾与完整回答）
        if len(input_ids) > max_length:
            input_ids = input_ids[-max_length:]
            attention_mask = attention_mask[-max_length:]
            labels = labels[-max_length:]

        samples.append(
            {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
        )
    return samples

def load_bts_qa_dataset(path, tokenizer, max_length):
    qa = load_and_inspect(path)
    samples = tokenize_function(qa, tokenizer=tokenizer, max_length=max_length)
    return samples

In [5]:
import os
import glob
model_repo = "Qwen/Qwen3-1.7B"
output_dir = "/kaggle/working"
max_length = 1024
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4
num_train_epochs = 2
learning_rate = 1e-4
save_steps = 400
logging_steps = 10
eval_steps = 200

# clear the files in /kaggle/working
folder_path = "/kaggle/working"
# Get a list of all files in the directory
files = glob.glob(os.path.join(folder_path, '*'))

for f in files:
    if os.path.isfile(f):
        os.remove(f)
print("All files deleted.")

All files deleted.


In [6]:
    
tokenizer = AutoTokenizer.from_pretrained(
    model_repo, use_fast=False, trust_remote_code=True
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_repo,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model.enable_input_require_grads()
samples = load_bts_qa_dataset("/kaggle/input/datasets/patrickhuang00017/bts-question-answer-dataset/qa.json", tokenizer, max_length=max_length)
train_dataset = Dataset.from_list(samples)
# print(samples)
# LoRA 配置（r=8）
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,
)
model = get_peft_model(model, lora_config)
kwargs = dict(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    logging_steps=logging_steps,
    save_steps=save_steps,
    eval_steps=eval_steps,
    gradient_checkpointing=True,
)
# 新增：每个 epoch 结束保存一次 checkpoint
kwargs["save_strategy"] = "epoch"
training_args = TrainingArguments(**kwargs)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
print(">>> 开始训练...")
trainer.train()
print(">>> 结束训练...进行保存模型")

model.save_pretrained(os.path.join(output_dir, "lora_adapter"))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded dataset from /kaggle/input/datasets/patrickhuang00017/bts-question-answer-dataset/qa.json: 63 examples
Columns: ['question', 'answer', 'source', 'category']

First record:
- question: IPS系统的主要功能是什么？
- answer: IPS（BTS智能生产系统）将BHS瓦线及流水线生产操作进行统一化，提高设备联动性和企业效益。它采用3层C/S数据库结构设计，保证系统可靠性与稳定性，使用中间服务器处理数据和业务逻辑，以PGSQL作为底层数据库存储。
- source: usermanual.pdf - 第5页
- category: 系统概述

First QA pair:
{'question': 'IPS系统的主要功能是什么？', 'answer': 'IPS（BTS智能生产系统）将BHS瓦线及流水线生产操作进行统一化，提高设备联动性和企业效益。它采用3层C/S数据库结构设计，保证系统可靠性与稳定性，使用中间服务器处理数据和业务逻辑，以PGSQL作为底层数据库存储。'}
>>> 开始训练...


Step,Training Loss
10,3.919612
20,3.486349
30,3.228901


>>> 结束训练...进行保存模型


In [7]:
# how to load the peft model
# 1. load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_repo,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
peft_model = PeftModel.from_pretrained(base_model, model_id="/kaggle/working/lora_adapter")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [8]:
messages = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "瓦线工作人员有哪些?"}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
test_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = peft_model.generate(
        input_ids=test_inputs.input_ids,
        max_new_tokens=1024,
        temperature=0.9,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
response_ids = outputs[0][len(test_inputs.input_ids[0]):]
response = tokenizer.decode(response_ids, skip_special_tokens=True)
print(response.strip())

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


用于瓦线的工作人员有：瓦线组长、瓦线班长、瓦线副组长、瓦线副班长。
